# Bias Bounty GPU Training Notebook
Self-evolving pipeline with GPU-accelerated XGBoost + LightGBM

In [ ]:
import numpy as np, pandas as pd, xgboost as xgb, lightgbm as lgb
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold
from scipy.optimize import minimize
import json, os
print("GPU available:", xgb.build_info())
print("XGBoost version:", xgb.__version__)

In [ ]:
# Load features from Kaggle dataset
df = pd.read_parquet("/kaggle/input/bias-bounty-features/all_regions_enhanced_features.parquet")
print(f"Loaded: {df.shape}")

In [ ]:
# Prepare features
drop = ["GEOID","region","county_fips","state_fips","centroid_lat","centroid_lon",
        "building_gap","road_gap","building_ratio","road_ratio",
        "building_count_ratio","building_count_gap","road_count_ratio","road_count_gap",
        "road_length_ratio","road_length_gap","poi_facility_gap","poi_to_facility_ratio"]
fcols = [c for c in df.columns if c not in drop and df[c].dtype in [np.float64,np.float32,np.int64,np.int32,np.bool_]]
X = df[fcols].copy().fillna(-999)
y = df["building_gap"].copy()
geo = df["GEOID"].copy()
valid = y.notna()
X, y, geo = X[valid], y[valid], geo[valid]
std = X.std(); X = X[std[std > 0].index]
corr = X.corrwith(y).abs().sort_values(ascending=False)
X = X[corr.head(80).index.tolist()]
print(f"{X.shape[1]} features, {X.shape[0]} tracts")

In [ ]:
# GPU-accelerated XGBoost with spatial CV
groups = geo.str[:5]
gkf = GroupKFold(n_splits=5)
oof_xgb = np.full(len(y), np.nan)
for fi, (ti, vi) in enumerate(gkf.split(X, y, groups)):
    m = xgb.XGBRegressor(n_estimators=2000, max_depth=8, learning_rate=0.01,
                         subsample=0.8, colsample_bytree=0.7, reg_alpha=0.1, reg_lambda=1.0,
                         tree_method="gpu_hist", random_state=42)
    m.fit(X.iloc[ti], y.iloc[ti], eval_set=[(X.iloc[vi], y.iloc[vi])], verbose=100)
    oof_xgb[vi] = m.predict(X.iloc[vi])
    print(f"Fold {fi}: RMSE={np.sqrt(mean_squared_error(y.iloc[vi], oof_xgb[vi])):.6f}")
print(f"XGBoost GPU: RMSE={np.sqrt(mean_squared_error(y, oof_xgb)):.6f} R2={r2_score(y, oof_xgb):.4f}")

In [ ]:
# LightGBM with spatial CV
oof_lgb = np.full(len(y), np.nan)
for fi, (ti, vi) in enumerate(gkf.split(X, y, groups)):
    m = lgb.LGBMRegressor(n_estimators=2000, max_depth=8, num_leaves=63, learning_rate=0.01,
                         subsample=0.8, colsample_bytree=0.7, reg_alpha=0.1, reg_lambda=1.0,
                         device="gpu", verbose=-1, random_state=42)
    m.fit(X.iloc[ti], y.iloc[ti], eval_set=[(X.iloc[vi], y.iloc[vi])])
    oof_lgb[vi] = m.predict(X.iloc[vi])
    print(f"Fold {fi}: RMSE={np.sqrt(mean_squared_error(y.iloc[vi], oof_lgb[vi])):.6f}")
print(f"LightGBM GPU: RMSE={np.sqrt(mean_squared_error(y, oof_lgb)):.6f} R2={r2_score(y, oof_lgb):.4f}")

In [ ]:
# Optimal blend
mat = np.column_stack([oof_xgb, oof_lgb])
valid_m = ~np.any(np.isnan(mat), axis=1)
mv, yv = mat[valid_m], y.values[valid_m]
res = minimize(lambda w: np.sqrt(mean_squared_error(yv, mv@w)), [0.5,0.5],
              method="SLSQP", bounds=[(0,1),(0,1)], constraints={"type":"eq","fun":lambda w:sum(w)-1})
blended = mat @ res.x
print(f"Blend weights: XGB={res.x[0]:.3f}, LGB={res.x[1]:.3f}")
print(f"Blended RMSE={res.fun:.6f} R2={r2_score(y, blended):.4f}")

In [ ]:
# Save predictions
submission = pd.DataFrame({"GEOID": geo.values, "coverage_gap_score": blended})
submission.to_csv("submission.csv", index=False)
print(f"Saved {len(submission)} predictions")
print(submission.describe())